# 01 SentencePiece Tokenizer

Train Korean and English SentencePiece tokenizers for the seq2seq Transformer project. Generated `spm_*` files are saved under the repository `data/` directory.

## Colab Setup

Run the next cell first when using Google Colab. It clones the repository, installs project dependencies from `pyproject.toml`, and moves the working directory to the repository root.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/songhahyun/seq2seq_transformer_model.git"
REPO_DIR = Path("/content/seq2seq_transformer_model")

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", "dev", "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", ".[notebooks]"], check=True)
else:
    print("Not running in Colab; skipping clone/install.")

print(f"cwd: {Path.cwd()}")

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"repo root: {REPO_ROOT}")
print(f"data dir : {DATA_DIR}")

In [ ]:
from src.config import Config
from src.data_pipeline import (
    set_seed,
    load_ko_en_dataset,
    extract_pairs,
    split_pairs,
    maybe_take_subset,
    prepare_tokenizers,
    encode_text,
    decode_ids,
)

config = Config()
config.sp_model_prefix_src = (DATA_DIR / "spm_ko").as_posix()
config.sp_model_prefix_tgt = (DATA_DIR / "spm_en").as_posix()

set_seed(config.random_seed)

print(config.dataset_name)
print(config.sp_model_prefix_src)
print(config.sp_model_prefix_tgt)

## Load dataset

The configured dataset is expected to contain `ko` and `en` columns.

In [ ]:
dataset = load_ko_en_dataset(
    config.dataset_name,
    split=config.train_split,
    hf_token=config.hf_token,
)

pairs = extract_pairs(dataset, src_col="ko", tgt_col="en")
train_pairs, valid_pairs, test_pairs = split_pairs(
    pairs,
    valid_ratio=config.valid_ratio,
    test_ratio=config.test_ratio,
    seed=config.random_seed,
)

train_pairs = maybe_take_subset(train_pairs, config.train_subset_size)

print(f"total pairs: {len(pairs)}")
print(f"train pairs: {len(train_pairs)}")
print(train_pairs[0])

## Train or load tokenizers

If `data/spm_ko.model` and `data/spm_en.model` already exist, the existing tokenizer files are loaded. Delete the files first when retraining with changed vocabulary settings or different training data.

In [ ]:
sp_src, sp_tgt = prepare_tokenizers(train_pairs, config)

print(f"source vocab size: {sp_src.get_piece_size()}")
print(f"target vocab size: {sp_tgt.get_piece_size()}")

for path in sorted(DATA_DIR.glob("spm_*")):
    print(path.relative_to(REPO_ROOT))

## Smoke test

Encode and decode one source sentence to verify the generated tokenizer can be used.

In [ ]:
sample_src, sample_tgt = train_pairs[0]
sample_ids = encode_text(
    sp_src,
    sample_src,
    config.max_length,
    config.bos_id,
    config.eos_id,
)
decoded = decode_ids(
    sp_src,
    sample_ids,
    bos_id=config.bos_id,
    eos_id=config.eos_id,
    pad_id=config.pad_id,
)

print(f"source : {sample_src}")
print(f"target : {sample_tgt}")
print(f"ids    : {sample_ids[:20]}")
print(f"decoded: {decoded}")